# Open-Ended Review Entity & Sentiment Discovery

This notebook demonstrates a first-pass approach for discovering emergent entities from free-form reviews and ranking them by aggregated sentiment. It is intentionally lightweight and assumption-driven so it can be iterated on quickly. Steps:

1. Load reviews from `reviews.csv`.
2. Extract candidate entities (keyphrases) per review.
3. Estimate sentiment per review and assign it to the entities mentioned there.
4. Merge semantically similar entities via embedding clustering.
5. Aggregate sentiments per merged entity to rank reputations.

Assumptions & notes:
- Reviews are free-form text (can be multilingual). We use keyword extraction (YAKE) instead of language-specific NER to avoid schema assumptions.
- Sentiment is inferred at the review level with a multilingual model and propagated to its entities; per-entity sentiment extraction can be added later.
- Semantic merging uses multilingual sentence embeddings with agglomerative clustering; adjust the distance threshold to merge more or fewer entities.


In [ ]:
# Install dependencies (run once per environment)
%pip install --quiet pandas numpy sentence-transformers yake scikit-learn torch torchvision torchaudio transformers


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from transformers import pipeline
import yake

pd.set_option('display.max_colwidth', None)


In [ ]:
# Load reviews
df = pd.read_csv('reviews.csv', header=None, names=['review_text'])
df.head()


In [ ]:
# Extract candidate entities/keyphrases per review using YAKE

# YAKE works well for short documents; tune max_ngram_size to control phrase length
kw_extractor = yake.KeywordExtractor(lan='multi', n=3, top=5)

entity_rows = []
for idx, row in df.iterrows():
    text = str(row['review_text'])
    keywords = kw_extractor.extract_keywords(text)
    # keywords returns list of (phrase, score); lower score = more relevant
    for phrase, score in keywords:
        cleaned = phrase.strip().lower()
        if cleaned:
            entity_rows.append({'review_id': idx, 'entity': cleaned, 'yake_score': score})

entities_df = pd.DataFrame(entity_rows)
entities_df.head()


In [ ]:
# Sentiment analysis per review (multilingual). The model outputs 1-5 stars.
sentiment_model = pipeline('sentiment-analysis', model='nlptown/bert-base-multilingual-uncased-sentiment')

sentiment_rows = []
for idx, row in df.iterrows():
    text = str(row['review_text'])
    result = sentiment_model(text)[0]
    # label is like '4 stars'; convert to int
    stars = int(result['label'].split()[0])
    sentiment_rows.append({'review_id': idx, 'sentiment_score': stars, 'sentiment_label': result['label'], 'sentiment_confidence': result['score']})

sentiment_df = pd.DataFrame(sentiment_rows)
sentiment_df.head()


In [ ]:
# Merge entity mentions with review-level sentiment
mentions_df = entities_df.merge(sentiment_df[['review_id', 'sentiment_score']], on='review_id', how='left')
mentions_df.head()


In [ ]:
# Embed entities and cluster to merge similar mentions
# Using a multilingual model to capture cross-language similarity
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

unique_entities = mentions_df['entity'].unique().tolist()
embeddings = embedder.encode(unique_entities, convert_to_tensor=False)

# Clustering: distance_threshold controls how aggressively we merge (lower = stricter)
# Adjust 'distance_threshold' after inspecting results.
clustering = AgglomerativeClustering(
    n_clusters=None,
    affinity='cosine',
    linkage='average',
    distance_threshold=0.35,
)
labels = clustering.fit_predict(embeddings)

cluster_map = {
    entity: f"cluster_{label}"
    for entity, label in zip(unique_entities, labels)
}

mentions_df['entity_cluster'] = mentions_df['entity'].map(cluster_map)
mentions_df.head()


In [ ]:
# Aggregate sentiment per merged entity cluster
entity_stats = (
    mentions_df
    .groupby('entity_cluster')
    .agg(
        n_mentions=('entity', 'count'),
        unique_entities=('entity', lambda x: sorted(set(x))),
        mean_sentiment=('sentiment_score', 'mean'),
        median_sentiment=('sentiment_score', 'median'),
    )
    .reset_index()
    .sort_values(by='mean_sentiment', ascending=False)
)

entity_stats


## Next steps
- Improve entity extraction with language-specific tokenizers or NER models if the data distribution is known.
- Move from review-level to entity-level sentiment by pairing entity spans with sentence-level sentiment.
- Experiment with different clustering thresholds or algorithms (e.g., HDBSCAN) to better control entity merging.
- Persist intermediate artifacts (embeddings, clusters) for incremental updates on new reviews.
